# Generate text with the best Nemotron-CC model

Loads the best-performing checkpoint from this project's sweep
(`num_layers=24, num_heads=1, embed_dim=128`, single-head causal attention,
final training loss ≈1.39 nats/char) and uses it to autoregressively
continue a prompt, character by character.

Architecture matches `Simple_text_generation_from_scracth_multiGPU_nemotron_cc_singlehead.py`.

In [1]:
import torch
import torch.nn as nn

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device

device(type='cpu')

## Rebuild the character vocabulary

The checkpoint only stores model weights, not the char↔id mapping, so we rebuild it
deterministically from the same Nemotron-CC training text used at training time
(same top-256-frequent-chars + `<unk>` capping logic).

In [2]:
from datasets import load_dataset
from collections import Counter

dataset = load_dataset(
    "parquet",
    data_files={"train": "data/nemotron_cc/train.parquet"},
)
text = ''.join(dataset['train']['text'])

max_vocab_chars = 256
char_counts = Counter(text)
chars = [ch for ch, _ in char_counts.most_common(max_vocab_chars)]
unk_token = '<unk>'

stoi = {ch: i for i, ch in enumerate(chars)}
stoi[unk_token] = len(chars)
itos = {i: ch for ch, i in stoi.items()}
unk_id = stoi[unk_token]

encode = lambda s: [stoi.get(c, unk_id) for c in s]
decode = lambda l: ''.join([itos[i] for i in l])

print(f"vocab_size = {len(stoi)}")

Generating train split: 0 examples [00:00, ? examples/s]

vocab_size = 257


## Model definition (single-head, matches the training script)

In [3]:
def causal_mask(seq_len, device):
    return torch.triu(torch.ones(seq_len, seq_len, device=device, dtype=torch.bool), diagonal=1)


class AttentionBlock(nn.Module):
    """ One causal single-head self-attention layer with pre-LayerNorm and a residual
    connection, followed by a pre-LayerNorm MLP sublayer with its own residual connection. """

    def __init__(self, embed_dim, hidden_dim):
        super().__init__()
        self.ln = nn.LayerNorm(embed_dim)
        self.Wq = nn.Linear(embed_dim, embed_dim)
        self.Wk = nn.Linear(embed_dim, embed_dim)
        self.Wv = nn.Linear(embed_dim, embed_dim)
        self.Wproj = nn.Linear(embed_dim, embed_dim)
        self.ln2 = nn.LayerNorm(embed_dim)
        self.mlp = nn.Sequential(
            nn.Linear(embed_dim, hidden_dim),
            nn.GELU(),
            nn.Linear(hidden_dim, embed_dim),
        )

    def forward(self, x):
        B, T, C = x.shape
        x_norm = self.ln(x)
        Q = self.Wq(x_norm)
        K = self.Wk(x_norm)
        V = self.Wv(x_norm)
        scores = torch.matmul(Q, K.transpose(-2, -1)) / (C ** 0.5)
        scores = scores.masked_fill(causal_mask(T, x.device), float('-inf'))
        attn = torch.matmul(torch.softmax(scores, dim=-1), V)
        attn = self.Wproj(attn)
        x = x + attn
        x = x + self.mlp(self.ln2(x))
        return x


class MyNN(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_dim, seq_length, num_layers=1):
        super().__init__()
        self.token_embedding = nn.Embedding(vocab_size, embed_dim)
        self.position_embedding = nn.Embedding(seq_length, embed_dim)
        self.blocks = nn.ModuleList([AttentionBlock(embed_dim, hidden_dim) for _ in range(num_layers)])
        self.ln_f = nn.LayerNorm(embed_dim)
        self.Wo = nn.Linear(embed_dim, vocab_size)

    def forward(self, input_x):
        seq_len = input_x.shape[1]
        positions = torch.arange(seq_len, device=input_x.device)
        X = self.token_embedding(input_x) + self.position_embedding(positions)
        for block in self.blocks:
            X = block(X)
        X = self.ln_f(X)
        return self.Wo(X)

## Load the best checkpoint

`num_layers`, `embed_dim`, `hidden_dim`, and `seq_length` are all inferred from the
checkpoint's tensor shapes, so this cell doesn't need to hardcode them.

In [4]:
checkpoint_path = "model_weights_nemotron_cc_singlehead_seed42_20260819_nl24_lr5e-05_bs128_sl1024_ep15.pt"

state_dict = torch.load(checkpoint_path, map_location=device)

vocab_size, embed_dim = state_dict['token_embedding.weight'].shape
seq_length = state_dict['position_embedding.weight'].shape[0]
hidden_dim = state_dict['blocks.0.mlp.0.weight'].shape[0]
num_layers = len({k.split('.')[1] for k in state_dict if k.startswith('blocks.')})

assert vocab_size == len(stoi), "Checkpoint vocab_size doesn't match the rebuilt vocab -- wrong dataset?"

model = MyNN(vocab_size, embed_dim, hidden_dim, seq_length, num_layers=num_layers)
model.load_state_dict(state_dict)
model.to(device)
model.eval()

print(f"Loaded {checkpoint_path}")
print(f"vocab_size={vocab_size}, embed_dim={embed_dim}, hidden_dim={hidden_dim}, "
      f"seq_length={seq_length}, num_layers={num_layers}")

Loaded model_weights_nemotron_cc_singlehead_seed42_20260819_nl24_lr5e-05_bs128_sl1024_ep15.pt
vocab_size=257, embed_dim=128, hidden_dim=512, seq_length=1024, num_layers=24


## Generate text

Greedy decoding (`temperature=0`) reliably falls into repetition loops once the model
locks onto a locally-confident continuation -- `temperature=0.8` samples instead, which
avoids that at the cost of some coherence (this is a small from-scratch char-level model,
not a production LLM).

In [5]:
def generate(prompt, num_chars=200, temperature=0.8, seed=None):
    if seed is not None:
        torch.manual_seed(seed)

    context = encode(prompt)
    generated = []
    with torch.no_grad():
        for _ in range(num_chars):
            window = context[-seq_length:]
            window_tensor = torch.tensor(window, dtype=torch.long, device=device).unsqueeze(0)
            logits = model(window_tensor)
            next_logits = logits[0, -1]

            if temperature > 0:
                probs = torch.softmax(next_logits / temperature, dim=-1)
                next_id = torch.multinomial(probs, num_samples=1).item()
            else:
                next_id = torch.argmax(next_logits).item()

            generated.append(next_id)
            context.append(next_id)

    return decode(generated)

In [8]:
prompt = "The history of artificial intelligence began"
continuation = generate(prompt, num_chars=1000, temperature=0.8, seed=42)

print("Prompt:", prompt)
print("Continuation:", continuation)

Prompt: The history of artificial intelligence began
Continuation:  the default of the digital a piece of research ensuring as benefit for conservation in the content of personalizing to the dominate of business as a concern of the functional workshop began with the researchers, and original entity. Most of the team interpartick workers and the surface amount Legital with prior of the research ensuring which internal working to the Content Web Benefits and Sounded of the dominance with the courage ensures above to the older, the personalization work the surface that of the user to increased the Digital Intent and and the provide that begin makes the warming in the people users where sets that the benefit may flaws. This is takes in the luxury began research thin the native in the older the book web which finding at the digital for storage can be continued the concentration of the think him to give the second to an increase in the extense, that they were not seems to at they more not us